## Setting up data and directories

In [10]:
#!/usr/bin/env python
"""
Combined Group Power Analysis - Encoding
Combines BLAES and AMME encoding power analyses.
Generates outputs for each group (outputs/BLAES, outputs/AMME) and all combined (outputs/all).
"""

import os
import glob
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
#matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()
    
OUTPUT_BASE = os.path.join(SCRIPT_DIR, 'outputs')

POWER_RANGES = {'Theta': (4, 8), 'Slow gamma': (30, 55)}

ROI_COLORS_SPEC = {
    'BLA': 'teal', 'PRC': '#E61C59', 'EC': '#8A2BE2',
    'PHG': 'blue', 'CA': 'black', 'DG': 'skyblue', 'HPC': 'orange',
}

ROI_COLORS_BAR = {
    'PRC': '#FFC0CB', 'BLA': '#008080', 'EC': '#800080',
    'PHG': '#0000FF', 'CA': '#00CED1', 'DG': '#87CEEB', 'HPC': '#FFA500',
}

BLAES_ENCODING_REGION_EXCLUSIONS = {
    'BLA': {'BJH042', 'BJH029'},
}

AMME_ENCODING_REGION_EXCLUSIONS = {
    'BLA': {'amyg016', 'amyg046', 'amyg057', 'amyg037'},
}


# =========================================================================
#  HELPERS
# =========================================================================

def fix_region(df):
    df = df.copy()
    df['Region'] = df['Region'].replace('ER', 'EC')
    return df

def apply_subject_region_exclusions(df, exclude_map):
    if not exclude_map or 'Patient' not in df.columns or 'Region' not in df.columns:
        return df
    return df[~df.apply(
        lambda row: row['Region'] in exclude_map and row['Patient'] in exclude_map[row['Region']],
        axis=1,
    )]

def normalize_image_name(x):
    if pd.isna(x):
        return np.nan
    return os.path.basename(str(x).strip().replace('\\', '/')).lower()

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def merge_dicts(*dicts):
    """Merge multiple {roi: {subject: power_vector}} dicts."""
    merged = {}
    for d in dicts:
        for roi, sd in d.items():
            merged.setdefault(roi, {}).update(sd)
    return merged

def merge_sets(*sets_):
    merged = set()
    for s in sets_:
        merged.update(s or set())
    return merged

def sorted_freq_cols(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix)]
    return sorted(cols, key=lambda x: float(x.split(prefix)[1]))

def freq_vals(cols, prefix):
    return np.array([float(c.split(prefix)[1]) for c in cols], dtype=float)

def compute_diff_df(stim_d, nostim_d, freqs, ranges):
    """Compute per-subject stim-minus-nostim band power for bar plots."""
    rows = []
    all_rois = sorted(set(stim_d.keys()) | set(nostim_d.keys()))
    for roi in all_rois:
        if roi not in stim_d or roi not in nostim_d:
            continue
        for subj in sorted(set(stim_d[roi]) & set(nostim_d[roi])):
            spv, npv = stim_d[roi][subj], nostim_d[roi][subj]
            for bn, (lo, hi) in ranges.items():
                mask = (freqs >= lo) & (freqs <= hi)
                rows.append({'Patient': subj, 'Region': roi, 'power_range': bn,
                             'mean_power_diff': spv[mask].mean() - npv[mask].mean()})
    return pd.DataFrame(rows)

def collapse_power_across_memory_conditions(data):
    """Match notebook logic: average each subject's available stim x memory spectra."""
    collapsed = {}
    for key in ['nostim_rem', 'nostim_forg', 'stim_rem', 'stim_forg']:
        cond_data = data.get(key, {})
        for roi, subj_dict in cond_data.items():
            if roi.startswith('PNAS'):
                continue
            collapsed.setdefault(roi, {})
            for subject, vec in subj_dict.items():
                collapsed[roi].setdefault(subject, []).append(np.asarray(vec, dtype=np.float64))

    for roi, subj_dict in collapsed.items():
        for subject, vecs in subj_dict.items():
            collapsed[roi][subject] = np.nanmean(np.vstack(vecs), axis=0)
    return collapsed

def get_overall_power_for_plot(data):
    if data.get('use_memory_collapsed_overall'):
        collapsed = collapse_power_across_memory_conditions(data)
        if collapsed:
            return collapsed
    return data['group_all_power']

def normalize_trial_type_amme(ttype):
    if isinstance(ttype, str) and 'stim' in ttype.lower() and ttype.lower() != 'nostim':
        return 'stim'
    elif isinstance(ttype, str) and ttype.lower() == 'nostim':
        return 'nostim'
    return None



## Loading data

In [8]:
# =========================================================================
#  BLAES ENCODING LOADING
# =========================================================================

def load_blaes_encoding():
    project_path = '/Users/martinahollearn/Library/CloudStorage/Box-Box/InmanLab/BLAES_data/dissertation/LFP_analyses'
    data_path = os.path.join(project_path, 'Results_CSVOutput')

    power_files = glob.glob(os.path.join(data_path, "*phase1*Power*.csv"))
    phase3_files = glob.glob(os.path.join(data_path, "*phase3*Power*.csv"))

    # Build phase3 stimulation lookup
    lookup_parts = []
    for f in phase3_files:
        df3 = pd.read_csv(f)
        if not all(c in df3.columns for c in ['Patient', 'stimulus_code', 'stimulation']):
            continue
        img_col = next((c for c in ['full_im_name', 'full_img_name', 'imagename'] if c in df3.columns), None)
        if not img_col:
            continue
        sub = df3[['Patient', 'stimulus_code', img_col, 'stimulation']].copy()
        sub['img_key'] = sub[img_col].apply(normalize_image_name)
        sub = sub.dropna(subset=['Patient', 'stimulus_code', 'img_key'])
        sub = sub.drop_duplicates(subset=['Patient', 'stimulus_code', 'img_key'])
        lookup_parts.append(sub[['Patient', 'stimulus_code', 'img_key', 'stimulation']])

    phase3_lookup = (pd.concat(lookup_parts, ignore_index=True)
                     .drop_duplicates(subset=['Patient', 'stimulus_code', 'img_key'])
                     if lookup_parts else
                     pd.DataFrame(columns=['Patient', 'stimulus_code', 'img_key', 'stimulation']))

    print(f"  BLAES Phase 3 lookup: {len(phase3_lookup)} rows")

    data = {
        'group_all_power': {}, 'stim_power': {}, 'nostim_power': {},
        'bc_stim': {}, 'bc_nostim': {},
        'freqs_post': None, 'freqs_diff': None,
        'has_memory': False,
        'use_memory_collapsed_overall': False,
        'bc_bar_exclude_rois': {'HPC'},
        'bc_memory_collapsed_exclude_rois': set(),
    }

    for file in power_files:
        df = fix_region(pd.read_csv(file))
        if 'Patient' not in df.columns or 'Region' not in df.columns:
            continue

        # Add stimulation from phase3
        if 'imagename' in df.columns and 'stimulus_code' in df.columns:
            df['img_key'] = df['imagename'].apply(normalize_image_name)
            df['stimulus_code'] = pd.to_numeric(df['stimulus_code'], errors='coerce')
            phase3_lookup['stimulus_code'] = pd.to_numeric(phase3_lookup['stimulus_code'], errors='coerce')
            df = df.merge(phase3_lookup, on=['Patient', 'stimulus_code', 'img_key'], how='left')
            df['stimulation'] = pd.to_numeric(df['stimulation'], errors='coerce')
            df = df[df['stimulation'].isin([0, 1])]
        else:
            df['stimulation'] = np.nan

        df = apply_subject_region_exclusions(df, BLAES_ENCODING_REGION_EXCLUSIONS)
        if df.empty:
            continue

        freq_cols = sorted_freq_cols(df, 'post_Freq_')
        if not freq_cols:
            continue
        if data['freqs_post'] is None:
            data['freqs_post'] = freq_vals(freq_cols, 'post_Freq_')

        subject = df['Patient'].iloc[0]
        print(f"  BLAES encoding: {os.path.basename(file)} [{subject}]")

        # Overall power
        for _, row in df.groupby(['Patient', 'Region'])[freq_cols].mean().reset_index().iterrows():
            data['group_all_power'].setdefault(row['Region'], {})[row['Patient']] = row[freq_cols].values.astype(np.float64)

        # By stim condition
        for stim_val, key in [(0, 'nostim_power'), (1, 'stim_power')]:
            df_s = df[df['stimulation'] == stim_val]
            if df_s.empty:
                continue
            for _, row in df_s.groupby(['Patient', 'Region'])[freq_cols].mean().reset_index().iterrows():
                data[key].setdefault(row['Region'], {})[row['Patient']] = row[freq_cols].values.astype(np.float64)

        # Baseline-corrected
        diff_cols = sorted_freq_cols(df, 'diff_Freq_')
        if diff_cols:
            if data['freqs_diff'] is None:
                data['freqs_diff'] = freq_vals(diff_cols, 'diff_Freq_')
            for _, row in df.groupby(['Patient', 'Region', 'stimulation'])[diff_cols].mean().reset_index().iterrows():
                key = 'bc_stim' if int(row['stimulation']) == 1 else 'bc_nostim'
                data[key].setdefault(row['Region'], {})[row['Patient']] = row[diff_cols].values.astype(np.float64)

    return data


# =========================================================================
#  AMME ENCODING LOADING
# =========================================================================

def load_amme_encoding():
    project_path = '/Users/martinahollearn/Library/CloudStorage/Box-Box/InmanLab/AMME_Data_Emory/AMME_Data/LFP_analyses_Martina'
    data_path = os.path.join(project_path, 'Results_CSVOutput', 'Phase1')

    power_files = glob.glob(os.path.join(data_path, "*Power*.csv"))
    data = {
        'group_all_power': {}, 'stim_power': {}, 'nostim_power': {},
        'bc_stim': {}, 'bc_nostim': {},
        'freqs_post': None, 'freqs_diff': None,
        'has_memory': True,
        'use_memory_collapsed_overall': True,
        'bc_bar_exclude_rois': {'PHG'},
        'bc_memory_collapsed_exclude_rois': {'PHG'},
        'stim_rem': {}, 'stim_forg': {},
        'nostim_rem': {}, 'nostim_forg': {},
        'bc_stim_rem': {}, 'bc_stim_forg': {},
        'bc_nostim_rem': {}, 'bc_nostim_forg': {},
    }

    for file in power_files:
        df = pd.read_csv(file)
        if 'Patient' not in df.columns or 'Region' not in df.columns:
            continue

        df = apply_subject_region_exclusions(df, AMME_ENCODING_REGION_EXCLUSIONS)
        df['test_trial_type'] = df['test_trial_type'].apply(normalize_trial_type_amme)
        df = df[df['test_trial_type'].notnull()]
        if df.empty:
            continue

        df = df[df['test_yes_or_no'].isin(['yes', 'no'])]
        df['memory_cond'] = np.where(df['test_yes_or_no'] == 'yes', 'remembered', 'forgotten')

        freq_cols = sorted_freq_cols(df, 'post_Freq_')
        if not freq_cols:
            continue
        if data['freqs_post'] is None:
            data['freqs_post'] = freq_vals(freq_cols, 'post_Freq_')

        subject = df['Patient'].iloc[0]
        print(f"  AMME encoding: {os.path.basename(file)} [{subject}]")

        # Overall power
        for _, row in df.groupby(['Patient', 'Region'])[freq_cols].mean().reset_index().iterrows():
            data['group_all_power'].setdefault(row['Region'], {})[row['Patient']] = row[freq_cols].values.astype(np.float64)

        # By stim
        grouped = df.groupby(['Patient', 'Region', 'test_trial_type'])[freq_cols].mean().reset_index()
        for _, row in grouped[grouped['test_trial_type'] == 'stim'].iterrows():
            data['stim_power'].setdefault(row['Region'], {})[row['Patient']] = row[freq_cols].values.astype(np.float64)
        for _, row in grouped[grouped['test_trial_type'] == 'nostim'].iterrows():
            data['nostim_power'].setdefault(row['Region'], {})[row['Patient']] = row[freq_cols].values.astype(np.float64)

        # By stim x memory
        grouped_mem = df.groupby(['Patient', 'Region', 'test_trial_type', 'memory_cond'])[freq_cols].mean().reset_index()
        mem_key_map = {
            ('stim', 'remembered'): 'stim_rem', ('stim', 'forgotten'): 'stim_forg',
            ('nostim', 'remembered'): 'nostim_rem', ('nostim', 'forgotten'): 'nostim_forg',
        }
        for _, row in grouped_mem.iterrows():
            k = mem_key_map.get((row['test_trial_type'], row['memory_cond']))
            if k:
                data[k].setdefault(row['Region'], {})[row['Patient']] = row[freq_cols].values.astype(np.float64)

        # Baseline-corrected
        diff_cols = sorted_freq_cols(df, 'diff_Freq_')
        if diff_cols:
            if data['freqs_diff'] is None:
                data['freqs_diff'] = freq_vals(diff_cols, 'diff_Freq_')

            grouped_bc = df.groupby(['Patient', 'Region', 'test_trial_type'])[diff_cols].mean().reset_index()
            for _, row in grouped_bc[grouped_bc['test_trial_type'] == 'stim'].iterrows():
                data['bc_stim'].setdefault(row['Region'], {})[row['Patient']] = row[diff_cols].values.astype(np.float64)
            for _, row in grouped_bc[grouped_bc['test_trial_type'] == 'nostim'].iterrows():
                data['bc_nostim'].setdefault(row['Region'], {})[row['Patient']] = row[diff_cols].values.astype(np.float64)

            grouped_bc_mem = df.groupby(['Patient', 'Region', 'test_trial_type', 'memory_cond'])[diff_cols].mean().reset_index()
            bc_key_map = {
                ('stim', 'remembered'): 'bc_stim_rem', ('stim', 'forgotten'): 'bc_stim_forg',
                ('nostim', 'remembered'): 'bc_nostim_rem', ('nostim', 'forgotten'): 'bc_nostim_forg',
            }
            for _, row in grouped_bc_mem.iterrows():
                k = bc_key_map.get((row['test_trial_type'], row['memory_cond']))
                if k:
                    data[k].setdefault(row['Region'], {})[row['Patient']] = row[diff_cols].values.astype(np.float64)

    return data


## Plotting graphs

In [11]:
# =========================================================================
#  PLOTTING FUNCTIONS - COMMON (non-memory)
# =========================================================================

def plot_power_by_roi(data, out_dir, label):
    gap = get_overall_power_for_plot(data)
    freqs = data['freqs_post']
    if not gap or freqs is None:
        return
    fig, ax = plt.subplots(figsize=(12, 8))
    for roi in sorted(gap.keys()):
        if roi.startswith('PNAS'):
            continue
        mat = np.array(list(gap[roi].values()), dtype=np.float64)
        mean, std = mat.mean(0), mat.std(0)
        c = ROI_COLORS_SPEC.get(roi, 'gray')
        ax.plot(freqs, mean, color=c, label=f'{roi} ({mat.shape[0]})')
        ax.fill_between(freqs, mean - std, mean + std, alpha=0.2, color=c)
    ax.set_xlabel('Frequency (Hz)', fontsize=18, fontweight='bold')
    ax.set_ylabel('Power (dB)', fontsize=18, fontweight='bold')
    ax.set_title(f'{label} Encoding Group Power by ROI', fontsize=20, fontweight='bold')
    ax.tick_params(axis='both', labelsize=14)
    ax.legend(bbox_to_anchor=(1.02, 0.5), loc='center left', prop={'weight': 'bold', 'size': 12})
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.savefig(os.path.join(out_dir, 'GroupPower_byROI_encoding.png'), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:

# =========================================================================
#  PLOTTING FUNCTIONS - COMMON (non-memory)
# =========================================================================

def plot_power_by_roi(data, out_dir, label):
    gap = get_overall_power_for_plot(data)
    freqs = data['freqs_post']
    if not gap or freqs is None:
        return
    fig, ax = plt.subplots(figsize=(12, 8))
    for roi in sorted(gap.keys()):
        if roi.startswith('PNAS'):
            continue
        mat = np.array(list(gap[roi].values()), dtype=np.float64)
        mean, std = mat.mean(0), mat.std(0)
        c = ROI_COLORS_SPEC.get(roi, 'gray')
        ax.plot(freqs, mean, color=c, label=f'{roi} ({mat.shape[0]})')
        ax.fill_between(freqs, mean - std, mean + std, alpha=0.2, color=c)
    ax.set_xlabel('Frequency (Hz)', fontsize=18, fontweight='bold')
    ax.set_ylabel('Power (dB)', fontsize=18, fontweight='bold')
    ax.set_title(f'{label} Encoding Group Power by ROI', fontsize=20, fontweight='bold')
    ax.tick_params(axis='both', labelsize=14)
    ax.legend(bbox_to_anchor=(1.02, 0.5), loc='center left', prop={'weight': 'bold', 'size': 12})
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.savefig(os.path.join(out_dir, 'GroupPower_byROI_encoding.png'), dpi=300, bbox_inches='tight')
    plt.close()


def plot_power_by_patient(data, out_dir, label):
    gap = get_overall_power_for_plot(data)
    freqs = data['freqs_post']
    if not gap or freqs is None:
        return
    patient_power = {}
    for roi, sd in gap.items():
        for subj, pv in sd.items():
            patient_power.setdefault(subj, []).append(pv)
    fig, ax = plt.subplots(figsize=(12, 8))
    for subj in sorted(patient_power):
        mean_p = np.array(patient_power[subj]).mean(0)
        ax.plot(freqs, mean_p, label=subj)
    ax.set_xlabel('Frequency (Hz)', fontsize=18, fontweight='bold')
    ax.set_ylabel('Power (dB)', fontsize=18, fontweight='bold')
    ax.set_title(f'{label} Encoding Group Power by Patient', fontsize=20, fontweight='bold')
    ax.tick_params(axis='both', labelsize=14)
    ax.legend(bbox_to_anchor=(1.02, 0.5), loc='center left', prop={'weight': 'bold', 'size': 12})
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.savefig(os.path.join(out_dir, 'GroupPower_byPatient_encoding.png'), dpi=300, bbox_inches='tight')
    plt.close()


def plot_stim_vs_nostim(data, out_dir, label):
    freqs = data['freqs_post']
    sp, nsp = data['stim_power'], data['nostim_power']
    if not sp or not nsp or freqs is None:
        return
    all_rois = sorted(set(sp.keys()) | set(nsp.keys()))
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
    for ax, (title, rd) in zip(axes, [('No Stim', nsp), ('Stim', sp)]):
        for roi in all_rois:
            if roi not in rd or roi.startswith('PNAS'):
                continue
            mat = np.array(list(rd[roi].values()), dtype=np.float64)
            mean, std = mat.mean(0), mat.std(0)
            c = ROI_COLORS_SPEC.get(roi, 'gray')
            ax.plot(freqs, mean, color=c, label=f'{roi} ({mat.shape[0]})')
            ax.fill_between(freqs, mean - std, mean + std, alpha=0.2, color=c)
        ax.set_title(title, fontsize=18, fontweight='bold')
        ax.set_xlabel('Frequency (Hz)', fontsize=16, fontweight='bold')
        ax.tick_params(axis='both', labelsize=12)
    axes[0].set_ylabel('Power (dB)', fontsize=16, fontweight='bold')
    axes[1].legend(bbox_to_anchor=(1.02, 0.5), loc='center left', prop={'weight': 'bold', 'size': 11})
    fig.suptitle(f'{label} Encoding Group Power: No Stim vs Stim', fontsize=20, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.88, 0.95])
    plt.savefig(os.path.join(out_dir, 'GroupPower_byROI_encoding_stim_vs_nostim.png'), dpi=300, bbox_inches='tight')
    plt.close()


def plot_per_roi_patient_stim_nostim(data, out_dir, label):
    freqs = data['freqs_post']
    sp, nsp = data['stim_power'], data['nostim_power']
    if not sp or not nsp or freqs is None:
        return
    all_rois = sorted(set(sp.keys()) | set(nsp.keys()))
    for roi in all_rois:
        if roi.startswith('PNAS'):
            continue
        fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
        plotted_any = False
        for ax, (title, rd) in zip(axes, [(f'{roi} - No Stim', nsp), (f'{roi} - Stim', sp)]):
            if roi not in rd or len(rd[roi]) == 0:
                ax.set_title(title, fontsize=18, fontweight='bold')
                ax.set_xlabel('Frequency (Hz)', fontsize=16, fontweight='bold')
                ax.tick_params(axis='both', labelsize=12)
                continue
            for subj in sorted(rd[roi]):
                ax.plot(freqs, rd[roi][subj], label=subj)
            plotted_any = True
            ax.set_title(title, fontsize=18, fontweight='bold')
            ax.set_xlabel('Frequency (Hz)', fontsize=16, fontweight='bold')
            ax.tick_params(axis='both', labelsize=12)
            ax.legend(bbox_to_anchor=(1.02, 0.5), loc='center left', prop={'weight': 'bold', 'size': 10})
        axes[0].set_ylabel('Power (dB)', fontsize=16, fontweight='bold')
        fig.suptitle(f'{label} Encoding Power: {roi} by Patient', fontsize=20, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 0.88, 0.95])
        if plotted_any:
            plt.savefig(os.path.join(out_dir, f'GroupPower_{roi}_byPatient_encoding_stim_vs_nostim.png'), dpi=300, bbox_inches='tight')
        plt.close()


def plot_bc_bar_graph(data, out_dir, label):
    freqs = data['freqs_diff']
    bcs, bcn = data['bc_stim'], data['bc_nostim']
    if not bcs or not bcn or freqs is None:
        return
    df_diff = compute_diff_df(bcs, bcn, freqs, POWER_RANGES)
    if df_diff.empty:
        return
    df_diff = df_diff[~df_diff['Region'].str.startswith('PNAS')]
    exclude_rois = data.get('bc_bar_exclude_rois', set())
    if exclude_rois:
        df_diff = df_diff[~df_diff['Region'].isin(exclude_rois)]
    if df_diff.empty:
        return
    unique_rois = [r for r in df_diff['Region'].unique() if r in ROI_COLORS_BAR]
    if not unique_rois:
        return
    palette = {r: ROI_COLORS_BAR[r] for r in unique_rois}
    g = sns.catplot(
        data=df_diff, x='Region', y='mean_power_diff', col='power_range',
        kind='bar', hue='Region', errorbar='se', palette=palette, legend=False,
        order=unique_rois, col_order=sorted(df_diff['power_range'].unique())
    )
    for ax, pr in zip(g.axes.flat, sorted(df_diff['power_range'].unique())):
        sub = df_diff[df_diff['power_range'] == pr]
        sns.stripplot(data=sub, x='Region', y='mean_power_diff', ax=ax,
                      color='black', alpha=0.5, jitter=0.2, order=unique_rois)
    for ax in g.axes.flat:
        ax.set_xlabel('')
        ax.set_ylabel('Baseline-Corrected Power Diff', fontsize=16, fontweight='bold')
        ax.tick_params(axis='x', labelsize=14, width=2, length=6)
        ax.tick_params(axis='y', labelsize=14, width=2, length=6)
        for l in ax.get_xticklabels():
            l.set_fontsize(14)
            l.set_fontweight('bold')
        ax.axhline(0, linestyle='-', color='grey')
    g.set_titles('{col_name}', size=15, weight='bold')
    g.fig.suptitle(f'{label} Encoding Baseline-Corrected Power Diff (Stim - No Stim)',
                   fontsize=20, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'Bargraph_baseline_corrected_powerDiff_byROI_encoding.png'),
                bbox_inches='tight', dpi=300)
    plt.close()


def plot_bc_per_roi(data, out_dir, label):
    freqs = data['freqs_diff']
    bcs, bcn = data['bc_stim'], data['bc_nostim']
    if freqs is None:
        return
    all_rois = sorted(set(bcs.keys()) | set(bcn.keys()))
    for roi in all_rois:
        if roi.startswith('PNAS'):
            continue
        nd, sd = bcn.get(roi, {}), bcs.get(roi, {})
        if not nd and not sd:
            continue
        fig, ax = plt.subplots(figsize=(8, 6))
        for cond_name, cond_d, color in [('No Stim', nd, 'blue'), ('Stim', sd, 'red')]:
            if not cond_d:
                continue
            mat = np.array(list(cond_d.values()), dtype=np.float64)
            n = mat.shape[0]
            mean, se = mat.mean(0), mat.std(0) / np.sqrt(n)
            ax.plot(freqs, mean, color=color, label=f'{cond_name} (n={n})')
            ax.fill_between(freqs, mean - se, mean + se, alpha=0.2, color=color)
        ax.set_title(f'{label} Encoding {roi} Baseline-Corrected Power', fontsize=18, fontweight='bold')
        ax.set_xlabel('Frequency (Hz)', fontsize=16, fontweight='bold')
        ax.set_ylabel('Power (dB)', fontsize=16, fontweight='bold')
        ax.tick_params(axis='both', labelsize=14)
        ax.set_xlim(1, 100)
        ax.legend(fontsize=12)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'{roi}_BaselineCorrectedPower_encoding_stim_vs_nostim.png'), dpi=300)
        plt.close()


# =========================================================================
#  PLOTTING FUNCTIONS - MEMORY SPECIFIC
# =========================================================================

def plot_quadrant_stim_memory(data, out_dir, label):
    freqs = data['freqs_post']
    if freqs is None:
        return
    mem_dicts = {
        'NoStim Remembered': data.get('nostim_rem', {}),
        'NoStim Forgotten': data.get('nostim_forg', {}),
        'AvgStim Remembered': data.get('stim_rem', {}),
        'AvgStim Forgotten': data.get('stim_forg', {}),
    }
    if not any(mem_dicts.values()):
        return
    all_rois = sorted(set().union(*[d.keys() for d in mem_dicts.values()]))
    gspr = {}
    for cd in mem_dicts.values():
        for roi, sd in cd.items():
            gspr.setdefault(roi, set()).update(sd.keys())

    cond_order = ['NoStim Remembered', 'NoStim Forgotten', 'AvgStim Remembered', 'AvgStim Forgotten']
    fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex=True, sharey=True)
    axes_flat = axes.flatten()
    plotted = set()
    for ax, cn in zip(axes_flat, cond_order):
        cd = mem_dicts[cn]
        for roi in all_rois:
            if roi.startswith('PNAS') or roi not in cd:
                continue
            mat = np.array(list(cd[roi].values()), dtype=np.float64)
            mean, std = mat.mean(0), mat.std(0)
            c = ROI_COLORS_SPEC.get(roi, 'gray')
            lbl = None
            if roi not in plotted:
                lbl = f'{roi} ({len(gspr.get(roi, set()))})'
                plotted.add(roi)
            ax.plot(freqs, mean, color=c, label=lbl)
            ax.fill_between(freqs, mean - std, mean + std, alpha=0.2, color=c)
        ax.set_title(cn, fontsize=18, fontweight='bold')
        ax.tick_params(axis='both', labelsize=14)
    fig.text(0.5, 0.04, 'Frequency (Hz)', ha='center', fontsize=18, fontweight='bold')
    fig.text(0.04, 0.5, 'Power (dB)', va='center', rotation='vertical', fontsize=18, fontweight='bold')
    fig.suptitle(f'{label} Encoding Group Power by Stim x Memory', fontsize=20, fontweight='bold')
    handles, labels_ = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels_, bbox_to_anchor=(0.82, 0.5), loc='center left',
               prop={'weight': 'bold', 'size': 12})
    plt.tight_layout(rect=[0.06, 0.06, 0.82, 0.94])
    plt.savefig(os.path.join(out_dir, 'Quadrant_StimMemory_Power_encoding.png'), dpi=300)
    plt.close()


def plot_per_roi_quadrant(data, out_dir, label):
    freqs = data['freqs_post']
    if freqs is None:
        return
    qc = {
        'Stim Remembered': data.get('stim_rem', {}),
        'Stim Forgotten': data.get('stim_forg', {}),
        'NoStim Remembered': data.get('nostim_rem', {}),
        'NoStim Forgotten': data.get('nostim_forg', {}),
    }
    qo = {'Stim Remembered': (0, 0), 'Stim Forgotten': (0, 1),
          'NoStim Remembered': (1, 0), 'NoStim Forgotten': (1, 1)}
    all_rois = sorted(set().union(*[d.keys() for d in qc.values()]))
    for roi in all_rois:
        if roi.startswith('PNAS'):
            continue
        fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
        has_data = False
        for cn, cd in qc.items():
            r, c = qo[cn]
            ax = axes[r, c]
            if roi not in cd or not cd[roi]:
                ax.set_title(f'{cn} (no data)', fontsize=14, fontweight='bold')
                ax.set_axis_off()
                continue
            has_data = True
            for subj, pv in sorted(cd[roi].items()):
                ax.plot(freqs, pv, label=subj, alpha=0.7)
            ax.set_title(cn, fontsize=14, fontweight='bold')
            ax.tick_params(axis='both', labelsize=10)
            ax.legend(fontsize=8, loc='upper right')
        if not has_data:
            plt.close(fig)
            continue
        fig.suptitle(f'{label} Encoding {roi} — Individual by Stim x Memory', fontsize=18, fontweight='bold')
        fig.text(0.5, 0.04, 'Frequency (Hz)', ha='center', fontsize=14, fontweight='bold')
        fig.text(0.04, 0.5, 'Power (dB)', va='center', rotation='vertical', fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0.06, 0.06, 0.95, 0.92])
        plt.savefig(os.path.join(out_dir, f'IndivQuadrant_{roi}_StimMemory_encoding.png'), dpi=300)
        plt.close()


def plot_bc_bar_by_memory(data, out_dir, label):
    freqs = data['freqs_diff']
    if freqs is None:
        return
    mem_pairs = [
        ('remembered', data.get('bc_stim_rem', {}), data.get('bc_nostim_rem', {})),
        ('forgotten', data.get('bc_stim_forg', {}), data.get('bc_nostim_forg', {})),
    ]
    all_rows = []
    for mem, sd, nd in mem_pairs:
        df_diff = compute_diff_df(sd, nd, freqs, POWER_RANGES)
        if not df_diff.empty:
            df_diff['memory_cond'] = mem
            all_rows.append(df_diff)
    if not all_rows:
        return
    df_all = pd.concat(all_rows, ignore_index=True)
    df_all = df_all[~df_all['Region'].str.startswith('PNAS')]
    if df_all.empty:
        return

    # Collapsed across memory
    df_collapsed = df_all.groupby(['Patient', 'Region', 'power_range'], as_index=False)['mean_power_diff'].mean()
    exclude_rois = data.get('bc_memory_collapsed_exclude_rois', set())
    if exclude_rois:
        df_collapsed = df_collapsed[~df_collapsed['Region'].isin(exclude_rois)]
    if not df_collapsed.empty:
        unique_rois = [r for r in df_collapsed['Region'].unique() if r in ROI_COLORS_BAR]
        if unique_rois:
            palette = [ROI_COLORS_BAR.get(r, '#808080') for r in unique_rois]
            g = sns.catplot(data=df_collapsed, x='Region', y='mean_power_diff', col='power_range',
                            kind='bar', hue='Region', errorbar='se', palette=palette, legend=False, order=unique_rois)
            for ax, (pr, sub) in zip(g.axes.flat, df_collapsed.groupby('power_range')):
                sns.stripplot(data=sub, x='Region', y='mean_power_diff', ax=ax,
                              color='black', alpha=0.5, jitter=0.2, order=unique_rois)
            for ax in g.axes.flat:
                ax.set_xlabel('')
                ax.set_ylabel('Power Diff (Stim - NoStim)', fontsize=16, fontweight='bold')
                ax.tick_params(axis='x', labelsize=14, width=2, length=6)
                ax.tick_params(axis='y', labelsize=14, width=2, length=6)
                for l in ax.get_xticklabels():
                    l.set_fontweight('bold')
                ax.axhline(0, color='grey')
            g.set_titles('{col_name}', size=18, weight='bold')
            g.fig.suptitle(f'{label} Encoding Power Diff Collapsed Across Memory',
                           fontsize=20, fontweight='bold', y=1.05)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, 'Bargraph_Power_diff_collapsed_memory_encoding.png'),
                        bbox_inches='tight', dpi=300)
            plt.close()

    # Split by memory
    for mem, mem_label in [('remembered', 'Remembered'), ('forgotten', 'Forgotten')]:
        df_mem = df_all[df_all['memory_cond'] == mem]
        if df_mem.empty:
            continue
        unique_rois = [r for r in df_mem['Region'].unique() if r in ROI_COLORS_BAR]
        if not unique_rois:
            continue
        palette = [ROI_COLORS_BAR.get(r, '#808080') for r in unique_rois]
        g = sns.catplot(data=df_mem, x='Region', y='mean_power_diff', col='power_range',
                        kind='bar', hue='Region', errorbar='se', palette=palette, legend=False, order=unique_rois)
        for ax, (pr, sub) in zip(g.axes.flat, df_mem.groupby('power_range')):
            sns.stripplot(data=sub, x='Region', y='mean_power_diff', ax=ax,
                          color='black', alpha=0.5, jitter=0.2, order=unique_rois)
        for ax in g.axes.flat:
            ax.set_xlabel('')
            ax.set_ylabel('Power Diff', fontsize=16, fontweight='bold')
            ax.tick_params(axis='x', labelsize=14)
            ax.tick_params(axis='y', labelsize=14)
            for l in ax.get_xticklabels():
                l.set_fontweight('bold')
            ax.axhline(0, color='grey')
        g.set_titles('{col_name}', size=18, weight='bold')
        g.fig.suptitle(f'{label} Encoding Power Diff, {mem_label} Trials',
                       fontsize=20, fontweight='bold', y=1.05)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'Bargraph_Power_diff_{mem}_encoding.png'),
                    bbox_inches='tight', dpi=300)
        plt.close()


def plot_bc_remembered_forgotten(data, out_dir, label):
    freqs = data['freqs_diff']
    if freqs is None:
        return
    mem_to_dicts = {
        'Remembered': {'stim': data.get('bc_stim_rem', {}), 'nostim': data.get('bc_nostim_rem', {})},
        'Forgotten': {'stim': data.get('bc_stim_forg', {}), 'nostim': data.get('bc_nostim_forg', {})},
    }
    all_rois = set()
    for md in mem_to_dicts.values():
        all_rois.update(md['stim'].keys())
        all_rois.update(md['nostim'].keys())
    for roi in sorted(all_rois):
        if roi.startswith('PNAS'):
            continue
        has_roi = any(roi in mem_to_dicts[ml]['stim'] or roi in mem_to_dicts[ml]['nostim']
                      for ml in mem_to_dicts)
        if not has_roi:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
        for ax, ml in zip(axes, ['Remembered', 'Forgotten']):
            for cn, cd, color in [('nostim', mem_to_dicts[ml]['nostim'], 'blue'),
                                  ('stim', mem_to_dicts[ml]['stim'], 'red')]:
                if roi not in cd:
                    continue
                mat = np.array(list(cd[roi].values()), dtype=np.float64)
                n = mat.shape[0]
                mean, se = mat.mean(0), mat.std(0) / np.sqrt(n)
                ax.plot(freqs, mean, color=color, label=f'{cn} ({n})')
                ax.fill_between(freqs, mean - se, mean + se, alpha=0.2, color=color)
            ax.set_title(f'{ml} Trials', fontsize=16, fontweight='bold')
            ax.set_xlabel('Frequency (Hz)', fontsize=16, fontweight='bold')
            ax.tick_params(axis='both', labelsize=14)
            ax.set_xlim(1, 100)
        axes[0].set_ylabel('Power (dB)', fontsize=16, fontweight='bold')
        handles, labels_ = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels_, loc='center left', bbox_to_anchor=(0.90, 0.5), fontsize=12)
        fig.suptitle(f'{label} Encoding {roi} Baseline-Corrected Power', fontsize=18, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 0.92, 0.94])
        plt.savefig(os.path.join(out_dir, f'{roi}_Baseline_Adjusted_Power_RememberedForgotten_encoding.png'), dpi=300)
        plt.close()


# =========================================================================
#  MAIN
# =========================================================================

def generate_common_plots(data, out_dir, label):
    ensure_dir(out_dir)
    print(f"\n  Generating common plots for {label}...")
    plot_power_by_roi(data, out_dir, label)
    plot_power_by_patient(data, out_dir, label)
    plot_stim_vs_nostim(data, out_dir, label)
    plot_per_roi_patient_stim_nostim(data, out_dir, label)
    plot_bc_bar_graph(data, out_dir, label)
    plot_bc_per_roi(data, out_dir, label)


def generate_memory_plots(data, out_dir, label):
    if not data.get('has_memory'):
        return
    ensure_dir(out_dir)
    print(f"  Generating memory plots for {label}...")
    plot_quadrant_stim_memory(data, out_dir, label)
    plot_per_roi_quadrant(data, out_dir, label)
    plot_bc_bar_by_memory(data, out_dir, label)
    plot_bc_remembered_forgotten(data, out_dir, label)


if __name__ == '__main__':
    print("=" * 60)
    print("Combined Group Power Analysis - Encoding")
    print("=" * 60)

    print("\nLoading BLAES encoding data...")
    blaes = load_blaes_encoding()

    print("\nLoading AMME encoding data...")
    amme = load_amme_encoding()

    # Per-group plots
    blaes_dir = ensure_dir(os.path.join(OUTPUT_BASE, 'BLAES'))
    amme_dir = ensure_dir(os.path.join(OUTPUT_BASE, 'AMME'))
    all_dir = ensure_dir(os.path.join(OUTPUT_BASE, 'all'))

    generate_common_plots(blaes, blaes_dir, 'BLAES')
    generate_common_plots(amme, amme_dir, 'AMME')
    generate_memory_plots(amme, amme_dir, 'AMME')

    # Merge for "all"
    print("\nMerging data for all-combined analysis...")

    # Check frequency compatibility
    all_freqs_post = None
    if blaes['freqs_post'] is not None and amme['freqs_post'] is not None:
        if np.array_equal(blaes['freqs_post'], amme['freqs_post']):
            all_freqs_post = blaes['freqs_post']
        else:
            print("  WARNING: Different post frequencies between groups. Using BLAES frequencies for 'all'.")
            all_freqs_post = blaes['freqs_post']
    else:
        all_freqs_post = blaes['freqs_post'] if blaes['freqs_post'] is not None else amme['freqs_post']

    all_freqs_diff = None
    if blaes['freqs_diff'] is not None and amme['freqs_diff'] is not None:
        if np.array_equal(blaes['freqs_diff'], amme['freqs_diff']):
            all_freqs_diff = blaes['freqs_diff']
        else:
            print("  WARNING: Different diff frequencies between groups. Using BLAES frequencies for 'all'.")
            all_freqs_diff = blaes['freqs_diff']
    else:
        all_freqs_diff = blaes['freqs_diff'] if blaes['freqs_diff'] is not None else amme['freqs_diff']

    all_data = {
        'group_all_power': merge_dicts(blaes['group_all_power'], amme['group_all_power']),
        'stim_power': merge_dicts(blaes['stim_power'], amme['stim_power']),
        'nostim_power': merge_dicts(blaes['nostim_power'], amme['nostim_power']),
        'bc_stim': merge_dicts(blaes['bc_stim'], amme['bc_stim']),
        'bc_nostim': merge_dicts(blaes['bc_nostim'], amme['bc_nostim']),
        'freqs_post': all_freqs_post,
        'freqs_diff': all_freqs_diff,
        'has_memory': False,  # Only AMME has memory for encoding; skip for 'all' to avoid redundancy
        'use_memory_collapsed_overall': False,
        'bc_bar_exclude_rois': merge_sets(blaes.get('bc_bar_exclude_rois'), amme.get('bc_bar_exclude_rois')),
        'bc_memory_collapsed_exclude_rois': merge_sets(
            blaes.get('bc_memory_collapsed_exclude_rois'),
            amme.get('bc_memory_collapsed_exclude_rois'),
        ),
    }

    generate_common_plots(all_data, all_dir, 'All')

    print("\n" + "=" * 60)
    print("Done! Encoding outputs saved to:", OUTPUT_BASE)
    print("=" * 60)